# Work In Progress

In [ ]:
!pip install uv
!uv pip install  -r requirements.txt 

#new library
!pip install mlxtend


Restart Kernel here

Then we load in packages

In [ ]:
## import packages

import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Data manipulation and analysis
import numpy as np
import pandas as pd
from IPython.display import display

# Multi-dimensional arrays and datasets (e.g., NetCDF, Zarr)
import xarray as xr

# Geospatial raster data handling with CRS support
import rioxarray as rxr

# Raster operations and spatial windowing
import rasterio
from rasterio.windows import Window

# Feature preprocessing and data splitting
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.spatial import cKDTree

# Machine Learning
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load
from pystac.extensions.eo import EOExtension as eo

from datetime import date
from tqdm import tqdm
import os 

#NEW PACKAGES
import planetary_computer 
import dask 
from scipy import stats
from datetime import datetime
from dask.distributed import Client
from scipy.special import inv_boxcox

from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LassoCV
from sklearn.cluster import KMeans
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import RFECV
from sklearn.svm import SVR
from sklearn.model_selection import KFold
from sklearn.model_selection import GroupKFold
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingRegressor

import statsmodels.api as sm

import xgboost as xgb


import re

from mlxtend.feature_selection import SequentialFeatureSelector as SFS


def run_groupkfold_cv(X, y, groups, n_splits=5, param_name="Parameter"):
    gkf = GroupKFold(n_splits=n_splits)
    fold_results = []

    for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
        # print(f"\n=== Fold {fold+1} ===")

        # Split
        X_train, X_test = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[val_idx]

        # Scale
        X_train_scaled, X_test_scaled, scaler = scale_data(X_train, X_test)

        # Train
        model = train_model(X_train_scaled, y_train)

        # Evaluate (in-sample)
        y_train_pred, r2_train, rmse_t
        # Evaluate (out-sample)rain = evaluate_model(model, X_train_scaled, y_train, "Train")

        y_test_pred, r2_test, rmse_test = evaluate_model(model, X_test_scaled, y_test, "Test")

        fold_results.append((r2_train, rmse_train, r2_test, rmse_test))

    df_results_kfold = pd.DataFrame(fold_results, columns=['R2_Train', 'RMSE_Train', 'R2_Test', 'RMSE_Test']).reset_index().rename(columns={"index": "fold"})
    df_results_kfold['Parameter'] = param_name
    df_results_kfold['Features'] = ', '.join([col for col in X.columns if col != 'sample_location_group'])
    df_results_kfold = df_results_kfold[['Parameter', 'Features', 'R2_Train', 'RMSE_Train', 'R2_Test', 'RMSE_Test']]

    return df_results_kfold



try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False

# Set display options to show all rows
pd.set_option('display.max_rows', None)
# Set display options to show all columns
pd.set_option('display.max_columns', None)
print(df)    


## Loading in Data

In [ ]:
Water_Quality_df = pd.read_csv("data/water_quality_training_dataset.csv")
landsat_train_features = pd.read_csv("data/landsat_features_training_combined.csv")
Terraclimate_df = pd.read_csv("data/terraclimate_features_training_combined.csv")
sawater_df = pd.read_csv("data/depWaterAndSan/depWaterAndSan_features_training_chemReadings_nonnull.csv")

#Clean Up the Data
#Capitalize Col Names
def capitalize_words_keep_spacing(col):
    # Split on space or underscore but keep the separators
    parts = re.split(r'([ _])', col)
    # Capitalize text parts, keep separators unchanged
    return ''.join(p.title() if p not in [' ', '_'] else p for p in parts)
Terraclimate_df.columns = [capitalize_words_keep_spacing(c) for c in Terraclimate_df.columns]
landsat_train_features.columns = [capitalize_words_keep_spacing(c) for c in landsat_train_features.columns]
Water_Quality_df.columns = [capitalize_words_keep_spacing(c) for c in Water_Quality_df.columns]
sawater_df.columns = [capitalize_words_keep_spacing(c) for c in sawater_df.columns]

# Ensure consistent date type before merge
Terraclimate_df["Sample Date"] = pd.to_datetime(Terraclimate_df["Sample Date"], format="%Y-%m-%d", errors="coerce")
for df in [Water_Quality_df, landsat_train_features, Terraclimate_df, sawater_df]:
    df["Sample Date"] = pd.to_datetime(df["Sample Date"], format="%d-%m-%Y", errors="coerce")
    
#removing duplicate cols
Terraclimate_df=Terraclimate_df.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
landsat_train_features=landsat_train_features.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])

def add_formatted_join_column(df: pd.DataFrame, join_col_name, drop_cols=None)  -> pd.DataFrame:
    df['sample date join'] = pd.to_datetime(df['Sample Date'], format='%d-%m-%Y').dt.strftime('%d-%m-%Y')
    df['latitude join'] = df['Latitude'].round(6)
    df['longitude join'] = df['Longitude'].round(6)
    df[join_col_name] = df['latitude join'].astype(str) + "~" + df['longitude join'].astype(str) + "~" + df['sample date join']
    return df
    
Water_Quality_df = add_formatted_join_column(Water_Quality_df, "joincol")
landsat_train_features = add_formatted_join_column(landsat_train_features, "joincol")
Terraclimate_df = add_formatted_join_column(Terraclimate_df, "joincol")
sawater_df = add_formatted_join_column(sawater_df, "joincol")

landsat_train_features=landsat_train_features.drop(columns=['Latitude', 'Longitude', 'Sample Date', 'sample date join', 'latitude join', 'longitude join'])
Terraclimate_df=Terraclimate_df.drop(columns=['Latitude', 'Longitude', 'Sample Date', 'sample date join', 'latitude join', 'longitude join'])
sawater_df=sawater_df.drop(columns=['Latitude', 'Longitude', 'Sample Date', 'sample date join', 'latitude join', 'longitude join'])

wq_data = (
    Water_Quality_df
    .merge(landsat_train_features, on=["joincol"], how="inner")
    .merge(Terraclimate_df, on=["joincol"], how="inner")
    .merge(sawater_df, on=["joincol"], how="inner")
)       

wq_data=wq_data.drop(columns=['sample date join', 'latitude join', 'longitude join', 'joincol'])

'''
def combine_two_datasets(dataset1,dataset2,dataset3,dataset4):
    
    data = pd.concat([dataset1,dataset2,dataset3,dataset4], axis=1)
    data = data.loc[:, ~data.columns.duplicated()]
    return data

wq_data = combine_two_datasets(Water_Quality_df, landsat_train_features, Terraclimate_df, sawater_df)
'''
#Data type corrections 
wq_data['Sample Date'] = pd.to_datetime(wq_data['Sample Date'],  format='%d-%m-%Y')

def convert_text_cols_to_float(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        if (pd.api.types.is_object_dtype(out[c]) 
            or pd.api.types.is_string_dtype(out[c]) 
            or pd.api.types.is_categorical_dtype(out[c])):
            s = out[c].astype(str).str.replace('\u00A0', ' ', regex=False).str.strip()
            s = s.str.replace('\u2212', '-', regex=False)                 # unicode minus
            s = s.str.replace(r'^\(\s*(.*)\s*\)$', r'-\1', regex=True)    # (123) -> -123
            s = s.str.replace(r'^\s*([+]?\s*[\d, .]+)\s*-$', r'-\1', regex=True)  # 123- -> -123
            s = s.str.replace(r'(?<=\d),(?=\d{3}\b)', '', regex=True)      # thousands comma
            out[c] = pd.to_numeric(s, errors='coerce')                     # float64 by default with NaNs
    return out

wq_data = convert_text_cols_to_float(wq_data)

In [ ]:
def add_spatial_clusters(
    df: pd.DataFrame,
    lat_col: str = "Latitude",
    lon_col: str = "Longitude",
    n_clusters: int = 5,
    group_col: str = "spatial_group",
    random_state: int = 88
) -> pd.DataFrame:
    """
    Create spatial groups by clustering latitude/longitude.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe.
    lat_col : str
        Name of latitude column.
    lon_col : str
        Name of longitude column.
    n_clusters : int
        Number of spatial clusters to create.
    group_col : str
        Name of output cluster/group column.
    random_state : int
        Random seed for reproducibility.

    Returns
    -------
    pd.DataFrame
        Copy of df with an added spatial grouping column.
    """

    out = df.copy()

    # Ensure numeric coordinates
    out[lat_col] = pd.to_numeric(out[lat_col], errors="coerce")
    out[lon_col] = pd.to_numeric(out[lon_col], errors="coerce")

    # Identify rows with valid coordinates
    valid_mask = out[lat_col].notna() & out[lon_col].notna()

    if valid_mask.sum() == 0:
        raise ValueError("No valid latitude/longitude pairs found for clustering.")

    # Extract valid coordinates
    lat = out.loc[valid_mask, lat_col].to_numpy()
    lon = out.loc[valid_mask, lon_col].to_numpy()

    # Convert degrees to approximate km coordinates
    # Latitude: ~111.32 km per degree
    # Longitude: scaled by cos(latitude)
    lat_km = lat * 111.32
    lon_km = lon * 111.32 * np.cos(np.radians(lat))

    coords_km = np.column_stack([lat_km, lon_km])

    # Fit clustering
    km = KMeans(
        n_clusters=n_clusters,
        random_state=random_state,
        n_init=20
    )
    labels = km.fit_predict(coords_km)

    # Assign labels back to dataframe
    out[group_col] = np.nan
    out.loc[valid_mask, group_col] = labels
    out[group_col] = out[group_col].astype("Int64")

    return out

wq_data = add_spatial_clusters(
    wq_data,
    lat_col="Latitude",
    lon_col="Longitude",
    n_clusters=5,
    group_col="spatial_group"
)

print(wq_data.info())
def plot_spatial_clusters(
    df: pd.DataFrame,
    lat_col: str = "Latitude",
    lon_col: str = "Longitude",
    group_col: str = "spatial_group"
):
    plot_df = df.dropna(subset=[lat_col, lon_col, group_col]).copy()

    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(
        plot_df[lon_col],
        plot_df[lat_col],
        c=plot_df[group_col],
        cmap="tab10",
        s=20,
        alpha=0.8
    )
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title("Spatial Clusters")
    plt.colorbar(scatter, label=group_col)
    plt.show()

plot_spatial_clusters(wq_data)

In [ ]:

for tgt in ["Total Alkalinity", "Electrical Conductance", "Dissolved Reactive Phosphorus"]:
    print(f"\n=== {tgt} by spatial group ===")
    print(
        wq_data.groupby("spatial_group")[tgt]
        .agg(["count", "mean", "median", "std", "min", "max"])
        .sort_index()
    )


## Building Pipeline



### Configuration



In [ ]:

# =========================
# CONFIG
# =========================

TARGETS = [
    "Total Alkalinity",
    "Electrical Conductance",
    "Dissolved Reactive Phosphorus"
]

CHEM_COLS = [
    "Mg_Diss_Water",
    "Cl_Diss_Water",
    "Ca_Diss_Water",
    "No3_No2_N_Diss_Water",
    "Ph_Diss_Water",
    "Nh4_N_Diss_Water",
    "Na_Diss_Water",
    "Po4_P_Diss_Water",
    "So4_Diss_Water",
    "F_Diss_Water",
    "Tal_Diss_Water",
    "Ec_Phys_Water"
]

# Predictors that often benefit from log1p if non-negative
LOG1P_PREDICTOR_CANDIDATES = [
    "Mg_Diss_Water", "Cl_Diss_Water", "Ca_Diss_Water",
    "No3_No2_N_Diss_Water", "Ph_Diss_Water", "Nh4_N_Diss_Water",
    "Na_Diss_Water", "Po4_P_Diss_Water", "So4_Diss_Water",
    "F_Diss_Water", "Tal_Diss_Water", "Ec_Phys_Water",
    "Q", "Ppt_Roll12", "Pet"
]

# Target transforms for regression
TARGET_TRANSFORM_MAP = {
    "Total Alkalinity": None,
    "Electrical Conductance": "log1p",
    "Dissolved Reactive Phosphorus": "log1p"
}

# Optional DRP threshold for classification
DRP_THRESHOLD = 30.0




### Feature Engineering

In [ ]:

def add_time_features(df: pd.DataFrame, date_col: str = "Sample Date") -> pd.DataFrame:
    out = df.copy()

    if date_col in out.columns:
        out[date_col] = pd.to_datetime(out[date_col], dayfirst=True, errors="coerce")
        month = out[date_col].dt.month

        out["Month_sine"] = np.sin(2 * np.pi * month / 12.0)
        out["Month_cosine"] = np.cos(2 * np.pi * month / 12.0)

    return out


In [ ]:

def make_drop_list_v2(
    df: pd.DataFrame,
    target: str,
    group_col: str = "spatial_group",
    keep_chemistry: bool = True,
    keep_coords: bool = True
):
    drop_cols = []

    # --------------------------------------------------------
    # 1) Exact target leakage twins
    # --------------------------------------------------------
    target_leak_map = {
        "Total Alkalinity": ["Tal_Diss_Water", "Tal_Diss_Water_Dl"],
        "Electrical Conductance": ["Ec_Phys_Water", "Ec_Phys_Water_Dl"],
        "Dissolved Reactive Phosphorus": ["Po4_P_Diss_Water", "Po4_P_Diss_Water_Dl"]
    }
    drop_cols += target_leak_map.get(target, [])

    # --------------------------------------------------------
    # 2) Drop all *_Dl columns
    # --------------------------------------------------------
    drop_cols += [c for c in df.columns if c.endswith("_Dl")]

    # --------------------------------------------------------
    # 3) Remove all targets from X
    # --------------------------------------------------------
    all_targets = [
        "Total Alkalinity",
        "Electrical Conductance",
        "Dissolved Reactive Phosphorus"
    ]
    drop_cols += [c for c in all_targets if c in df.columns]

    # --------------------------------------------------------
    # 4) Generic correlation-based drops
    # --------------------------------------------------------
    corr_drops = [
        # NDMI / urban / dryness family
        "Msi",
        "Ndbi",
        "Bsi",
        "Ui (Urban Index)",

        # Vegetation / ratio family
        "Gcvi",
        "Gndvi",
        "Green/Nir Ratio",
        "Green/Red Ratio",
        "Ndgi",
        "Osavi",
        "Savi",
        "Red/Nir Ratio",

        # Moisture / burn / SWIR
        "Mndwi",
        "Nbr",
        "Swir16",

        # Radiation
        "Urad",

        # Hydro / climate
        "Aet",
        "Def",
        "Vpd",
        "Ppt",
        "Ppt_Roll6"
    ]
    drop_cols += [c for c in corr_drops if c in df.columns]

    # --------------------------------------------------------
    # 5) Optionally remove chemistry
    # --------------------------------------------------------
    if not keep_chemistry:
        chem_cols = [
            "Mg_Diss_Water", "Cl_Diss_Water", "Ca_Diss_Water",
            "No3_No2_N_Diss_Water", "Ph_Diss_Water", "Nh4_N_Diss_Water",
            "Na_Diss_Water", "Po4_P_Diss_Water", "So4_Diss_Water",
            "F_Diss_Water", "Tal_Diss_Water", "Ec_Phys_Water"
        ]
        drop_cols += [c for c in chem_cols if c in df.columns]

    # --------------------------------------------------------
    # 6) Always remove helper grouping columns
    # --------------------------------------------------------
    # Remove raw date once time features are derived
    if "Sample Date" in df.columns:
        drop_cols.append("Sample Date")

    # Optionally remove coordinates
    if not keep_coords:
        for c in ["Latitude", "Longitude"]:
            if c in df.columns:
                drop_cols.append(c)

    # Deduplicate
    drop_cols = list(dict.fromkeys([c for c in drop_cols if c in df.columns]))

    return drop_cols


In [ ]:

def prepare_xy_groups_v2(
    df: pd.DataFrame,
    target: str,
    group_col: str = "spatial_group",
    ablation: str = "full",          # "full", "chem_only", "no_chem"
    keep_coords: bool = True,
    log1p_predictors: bool = True
):
    """
    Prepare X, y, groups for one target.
    """

    out = df.copy()
    out = add_time_features(out)

    # keep only rows with target and group
    out = out.dropna(subset=[target, group_col]).copy()
    out = out[out[group_col].notna()].copy()

    # Build initial drop list
    drop_cols = make_drop_list_v2(
        out,
        target=target,
        group_col=group_col,
        keep_chemistry=True,   # keep first, then ablate below
        keep_coords=keep_coords
    )

    X = out.drop(columns=drop_cols, errors="ignore").copy()
    y = out[target].copy()
    groups = out[group_col].copy()

    # numeric only
    X = X.select_dtypes(include=[np.number]).copy()

    # Apply ablation
    chem_in_X = [c for c in CHEM_COLS if c in X.columns]

    if ablation == "full":
        pass
    elif ablation == "chem_only":
        X = X[chem_in_X].copy()
    elif ablation == "no_chem":
        X = X.drop(columns=chem_in_X, errors="ignore").copy()
    else:
        raise ValueError("ablation must be one of: 'full', 'chem_only', 'no_chem'")

    # Optional log1p transform on skewed predictors
    if log1p_predictors:
        for col in LOG1P_PREDICTOR_CANDIDATES:
            if col in X.columns:
                series = X[col]
                if pd.api.types.is_numeric_dtype(series):
                    if series.dropna().shape[0] > 0 and series.min(skipna=True) >= 0:
                        X[col] = np.log1p(series)

    return X, y, groups


### Models


In [ ]:

def make_regression_models_v2(random_state: int = 88):
    models = {}

    # --------------------------------------------------
    # Dummy baseline
    # --------------------------------------------------
    models["dummy_mean"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DummyRegressor(strategy="mean"))
    ])

    # --------------------------------------------------
    # Optional: LassoCV (automatically tunes alpha)
    # This is often more useful than fixed-alpha Lasso
    # --------------------------------------------------
    models["lasso_cv"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LassoCV(
            alphas=np.logspace(-3, 1, 30),   # 0.001 to 10
            cv=3,                            # internal CV inside training fold
            max_iter=10000,
            random_state=random_state,
            n_jobs=-1
        ))
    ])

    # --------------------------------------------------
    # Conservative Random Forest
    # --------------------------------------------------
    models["random_forest"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=500,
            max_depth=8,
            min_samples_leaf=10,
            max_features=0.3,
            random_state=random_state,
            n_jobs=-1
        ))
    ])

    # --------------------------------------------------
    # Conservative HistGradientBoosting
    # --------------------------------------------------
    models["hist_gbm"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingRegressor(
            learning_rate=0.03,
            max_depth=3,
            max_iter=300,
            min_samples_leaf=50,
            l2_regularization=5.0,
            random_state=random_state
        ))
    ])

    # --------------------------------------------------
    # Optional XGBoost
    # --------------------------------------------------
    if XGB_AVAILABLE:
        models["xgboost"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", xgb.XGBRegressor(
                n_estimators=300,
                learning_rate=0.03,
                max_depth=3,
                min_child_weight=10,
                subsample=0.7,
                colsample_bytree=0.5,
                reg_alpha=1.0,
                reg_lambda=3.0,
                objective="reg:squarederror",
                random_state=random_state,
                n_jobs=-1
            ))
        ])

    return models


In [ ]:

def maybe_wrap_target_transform_v2(model, target: str):
    transform = TARGET_TRANSFORM_MAP.get(target, None)

    if transform == "log1p":
        return TransformedTargetRegressor(
            regressor=model,
            func=np.log1p,
            inverse_func=np.expm1
        )

    return model


In [ ]:

def run_grouped_cv_models_regression_v2(
    X: pd.DataFrame,
    y: pd.Series,
    groups: pd.Series,
    target_name: str,
    models: dict,
    n_splits: int = 5
):
    valid_mask = groups.notna()
    X = X.loc[valid_mask].copy()
    y = y.loc[valid_mask].copy()
    groups = groups.loc[valid_mask].copy()

    n_unique_groups = groups.nunique()
    if n_unique_groups < 2:
        raise ValueError(f"Need at least 2 groups; found {n_unique_groups}")
    n_splits = min(n_splits, n_unique_groups)

    gkf = GroupKFold(n_splits=n_splits)

    summary_rows = []
    fold_rows = []
    oof_predictions = {}

    for model_name, base_model in models.items():
        y_oof = pd.Series(index=y.index, dtype=float)

        for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=groups), start=1):
            X_train = X.iloc[train_idx].copy()
            X_test = X.iloc[test_idx].copy()
            y_train = y.iloc[train_idx].copy()
            y_test = y.iloc[test_idx].copy()
            g_test = groups.iloc[test_idx].copy()

            model = clone(base_model)
            model = maybe_wrap_target_transform_v2(model, target_name)

            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            y_oof.iloc[test_idx] = y_pred

            fold_rows.append({
                "target": target_name,
                "model": model_name,
                "fold": fold,
                "n_train": len(train_idx),
                "n_test": len(test_idx),
                "groups_held_out": sorted(g_test.unique().tolist()),
                "r2": r2_score(y_test, y_pred),
                "rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
                "mae": mean_absolute_error(y_test, y_pred)
            })

        summary_rows.append({
            "target": target_name,
            "model": model_name,
            "r2_oof": r2_score(y, y_oof),
            "rmse_oof": np.sqrt(mean_squared_error(y, y_oof)),
            "mae_oof": mean_absolute_error(y, y_oof)
        })

        oof_predictions[model_name] = pd.DataFrame({
            "y_true": y,
            "y_pred": y_oof,
            "group": groups
        })

    summary_df = pd.DataFrame(summary_rows).sort_values("r2_oof", ascending=False).reset_index(drop=True)
    fold_df = pd.DataFrame(fold_rows)

    return summary_df, fold_df, oof_predictions


In [ ]:

def plot_cv_summary(fold_df: pd.DataFrame, target_name: str):
    """
    Plot fold-wise R2 / RMSE / MAE by model.
    """
    metrics = ["r2", "rmse", "mae"]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"Fold-wise CV diagnostics - {target_name}", fontsize=14)

    for ax, metric in zip(axes, metrics):
        pivot = fold_df.pivot(index="fold", columns="model", values=metric)
        pivot.plot(marker="o", ax=ax)
        ax.set_title(metric.upper())
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


def plot_oof_diagnostics(oof_df: pd.DataFrame, target_name: str, model_name: str):
    """
    Observed vs predicted and residual diagnostics.
    """
    y_true = oof_df["y_true"].values
    y_pred = oof_df["y_pred"].values
    residuals = y_true - y_pred

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"OOF Diagnostics - {target_name} - {model_name}", fontsize=14)

    # 1) Observed vs Predicted
    axes[0].scatter(y_true, y_pred, alpha=0.5)
    lo = np.nanmin([y_true.min(), y_pred.min()])
    hi = np.nanmax([y_true.max(), y_pred.max()])
    axes[0].plot([lo, hi], [lo, hi], linestyle="--")
    axes[0].set_xlabel("Observed")
    axes[0].set_ylabel("Predicted")
    axes[0].set_title("Observed vs Predicted")
    axes[0].grid(True, alpha=0.3)

    # 2) Residuals vs Predicted
    axes[1].scatter(y_pred, residuals, alpha=0.5)
    axes[1].axhline(0, linestyle="--")
    axes[1].set_xlabel("Predicted")
    axes[1].set_ylabel("Residual (Observed - Predicted)")
    axes[1].set_title("Residuals vs Predicted")
    axes[1].grid(True, alpha=0.3)

    # 3) Residual histogram
    axes[2].hist(residuals, bins=30, alpha=0.8)
    axes[2].axvline(0, linestyle="--")
    axes[2].set_title("Residual Distribution")
    axes[2].set_xlabel("Residual")
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


def plot_group_residuals(oof_df: pd.DataFrame, target_name: str, model_name: str):
    """
    Plot median residual per spatial group.
    """
    tmp = oof_df.copy()
    tmp["residual"] = tmp["y_true"] - tmp["y_pred"]

    group_summary = tmp.groupby("group")["residual"].median().sort_values()

    plt.figure(figsize=(10, 4))
    group_summary.plot(kind="bar")
    plt.axhline(0, linestyle="--")
    plt.title(f"Median residual by spatial group - {target_name} - {model_name}")
    plt.ylabel("Median residual")
    plt.xlabel("Spatial group")
    plt.tight_layout()
    plt.show()


### Creating Pipeline


In [ ]:

def run_target_regression_pipeline_v2(
    df: pd.DataFrame,
    target: str,
    group_col: str = "spatial_group",
    ablation: str = "full",
    keep_coords: bool = True,
    log1p_predictors: bool = True,
    n_splits: int = 5,
    random_state: int = 88
):
    X, y, groups = prepare_xy_groups_v2(
        df=df,
        target=target,
        group_col=group_col,
        ablation=ablation,
        keep_coords=keep_coords,
        log1p_predictors=log1p_predictors
    )

    print(f"\n=== {target} | regression | ablation={ablation} ===")
    print("Rows:", len(X))
    print("Features:", X.shape[1])
    print("Groups:", groups.nunique())

    models = make_regression_models_v2(random_state=random_state)

    summary_df, fold_df, oof_predictions = run_grouped_cv_models_regression_v2(
        X=X,
        y=y,
        groups=groups,
        target_name=target,
        models=models,
        n_splits=n_splits
    )

    print("\nModel summary:")
    print(summary_df)

    return {
        "summary": summary_df,
        "folds": fold_df,
        "oof_predictions": oof_predictions,
        "X": X,
        "y": y,
        "groups": groups
    }


## Run Model for Targets:

In [ ]:

results_v2 = {}

for tgt in TARGETS:
    results_v2[tgt] = run_target_regression_pipeline_v2(
        df=wq_data,
        target=tgt,
        group_col="spatial_group",
        ablation="full",
        keep_coords=True,          # <-- important change
        log1p_predictors=True,     # <-- new
        n_splits=5,
        random_state=88
    )


In [ ]:
ablation_results_v2 = {}

for tgt in TARGETS:
    ablation_results_v2[tgt] = {}

    for abl in ["full", "chem_only", "no_chem"]:
        ablation_results_v2[tgt][abl] = run_target_regression_pipeline_v2(
            df=wq_data,
            target=tgt,
            group_col="spatial_group",
            ablation=abl,
            keep_coords=True,
            log1p_predictors=True,
            n_splits=5,
            random_state=88
        )


### Additional Diagnostics


In [ ]:

print(results_v2["Total Alkalinity"]["summary"])
print(results_v2["Electrical Conductance"]["summary"])
print(results_v2["Dissolved Reactive Phosphorus"]["summary"])

print(ablation_results_v2["Dissolved Reactive Phosphorus"]["summary"])


In [ ]:
#This code takes a long time to run, and should only be used for diagnostic purposes
'''
CHEM_COLS = [
    "Mg_Diss_Water", "Cl_Diss_Water", "Ca_Diss_Water",
    "No3_No2_N_Diss_Water", "Ph_Diss_Water", "Nh4_N_Diss_Water",
    "Na_Diss_Water", "Po4_P_Diss_Water", "So4_Diss_Water",
    "F_Diss_Water", "Tal_Diss_Water", "Ec_Phys_Water"
]

def prepare_xy_groups_ablation(df, target, group_col="spatial_group", ablation="full"):
    X, y, groups = prepare_xy_groups(
        df=df,
        target=target,
        group_col=group_col,
        keep_chemistry=True
    )

    chem_in_X = [c for c in CHEM_COLS if c in X.columns]

    if ablation == "chem_only":
        X = X[chem_in_X].copy()
    elif ablation == "no_chem":
        X = X.drop(columns=chem_in_X, errors="ignore").copy()
    elif ablation == "full":
        pass
    else:
        raise ValueError("ablation must be 'full', 'chem_only', or 'no_chem'")

    return X, y, groups

    
for tgt in ["Total Alkalinity", "Electrical Conductance", "Dissolved Reactive Phosphorus"]:
    for abl in ["full", "chem_only", "no_chem"]:
        X, y, groups = prepare_xy_groups_ablation(wq_data, tgt, ablation=abl)

        summary_df, fold_df, oof_predictions = run_grouped_cv_models(
            X=X,
            y=y,
            groups=groups,
            target_name=f"{tgt} [{abl}]",
            models=make_models(),
            n_splits=5
        )

        print(f"\n=== {tgt} | {abl} ===")
        print(summary_df)
'''
